# Recuperação de Fusão na Pesquisa de Documentos
## Visão Geral
Este código implementa um sistema Fusion Retrieval que combina busca de similaridade baseada em vetores com recuperação BM25 baseada em palavras-chave. A abordagem visa aproveitar os pontos fortes de ambos os métodos para melhorar a qualidade global e a relevância da recuperação de documentos.

## Motivação
Métodos tradicionais de recuperação muitas vezes dependem de compreensão semântica (baseada em vetor) ou correspondência de palavras-chave (BM25). Cada abordagem tem seus pontos fortes e fracos. A recuperação de fusão visa combinar esses métodos para criar um sistema de recuperação mais robusto e preciso que possa lidar com uma gama mais ampla de consultas de forma eficaz.

## Componentes-chave

1. Processamento e chunks de texto PDF
2. Criação de armazenamento de vetor usando incorporações FAISS e OpenAI
3. Criação de índice BM25 para recuperação baseada em palavras-chave
4. Função personalizada de recuperação de fusão que combina ambos os métodos

## Detalhes do método
Pré-processamento de documentos

1. O PDF é carregado e dividido em chunks usando RecursiveCharacterTextSplitter.
2. Chunks são limpos substituindo 't' por espaços (provavelmente abordando uma questão de formatação específica).

Criação da Loja Vector

1. As embeddings da OpenAI são usadas para criar representações vetoriais dos chunks de texto.
2. Uma vector store FAISS é criada a partir desses embeddings para uma busca de similaridade eficiente.

## Criação do Índice BM25

1. Um índice BM25 é criado a partir dos mesmos chunks de texto usados para o vector store.
2. Isso permite a recuperação baseada em palavras-chave ao lado do método baseado em vetores.

Função de Recuperação de Fusão

A função `fusion_retrieval` é o núcleo desta implementação:

1. É preciso uma consulta e executa tanto a recuperação baseada em vetores quanto BM25.
2. As pontuações de ambos os métodos são normalizadas para uma escala comum.
3. Uma combinação ponderada destes escores é calculada (controlada pelo parâmetro `alpha`).
4. Os documentos são classificados com base nas pontuações combinadas, e os resultados top-k são devolvidos.

## Benefícios desta abordagem
1. Melhor qualidade de recuperação: Ao combinar busca semântica e baseada em palavras-chave, o sistema pode capturar similaridade conceitual e correspondências exatas de palavras-chave.
2. Flexibilidade: O parâmetro `alpha` permite ajustar o equilíbrio entre busca de vetores e palavras-chave com base em casos de uso específicos ou tipos de consulta.
3. Robustness: A abordagem combinada pode lidar eficazmente com uma gama mais vasta de consultas, atenuando as deficiências dos métodos individuais.
4. Personalizabilidade: O sistema pode ser facilmente adaptado para usar diferentes lojas de vetores ou métodos de recuperação baseados em palavras-chave.

## Conclusão
Recuperação de fusão representa uma abordagem poderosa para a pesquisa de documentos que combina os pontos fortes da compreensão semântica e correspondência de palavras-chave. Ao alavancar os métodos de recuperação baseados em vetores e BM25, ele oferece uma solução mais abrangente e flexível para tarefas de recuperação de informações. Esta abordagem tem aplicações potenciais em vários campos onde tanto semelhança conceitual quanto relevância de palavras-chave são importantes, como pesquisa acadêmica, busca legal de documentos ou motores de busca de propósito geral.

<div style="text-align: center;">

<img src="../images/fusion_retrieval.svg" alt="Fusion Retrieval" style="width:100%; height:auto;">
</div>

# Instalação e Importações do Pacote
A célula abaixo instala todos os pacotes necessários para executar este notebook.
.

In [ ]:
# Install required packages
!pip install langchain numpy python-dotenv rank-bm25

In [ ]:
# Clone the repository to access helper functions and evaluation modules
!git clone https://github.com/NirDiamant/RAG_TECHNIQUES.git
import sys
sys.path.append('RAG_TECHNIQUES')
# If you need to run with the latest data
# !cp -r RAG_TECHNIQUES/data .

In [ ]:
import os
import sys
from dotenv import load_dotenv
from langchain.docstore.document import Document

from typing import List
from rank_bm25 import BM25Okapi
import numpy as np


# Original path append replaced for Colab compatibility
from helper_functions import *
from evaluation.evalute_rag import *

# Load environment variables from a .env file
load_dotenv()

# Set the OpenAI API key environment variable
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

### Definir o caminho do documento

In [ ]:
# Download required data files
import os
os.makedirs('data', exist_ok=True)

# Download the PDF document used in this notebook
!wget -O data/Understanding_Climate_Change.pdf https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/Understanding_Climate_Change.pdf
!wget -O data/Understanding_Climate_Change.pdf https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/Understanding_Climate_Change.pdf


In [2]:
path = "data/Understanding_Climate_Change.pdf"

### Codifique o pdf para armazenar vetor e retorne o documento dividido do passo anterior para criar a instância BM25

In [3]:
def encode_pdf_and_get_split_documents(path, chunk_size=1000, chunk_overlap=200):
    """
    Encodes a PDF book into a vector store using OpenAI embeddings.

    Args:
        path: The path to the PDF file.
        chunk_size: The desired size of each text chunk.
        chunk_overlap: The amount of overlap between consecutive chunks.

    Returns:
        A FAISS vector store containing the encoded book content.
    """

    # Load PDF documents
    loader = PyPDFLoader(path)
    documents = loader.load()

    # Split documents into chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len
    )
    texts = text_splitter.split_documents(documents)
    cleaned_texts = replace_t_with_space(texts)

    # Create embeddings and vector store
    embeddings = OpenAIEmbeddings()
    vectorstore = FAISS.from_documents(cleaned_texts, embeddings)

    return vectorstore, cleaned_texts

### Criar vectorstore e obter os documentos em bloco

In [4]:
vectorstore, cleaned_texts = encode_pdf_and_get_split_documents(path)

## Criar um índice bm25 para recuperar documentos por palavras-chave

In [5]:
def create_bm25_index(documents: List[Document]) -> BM25Okapi:
    """
    Create a BM25 index from the given documents.

    BM25 (Best Matching 25) is a ranking function used in information retrieval.
    It's based on the probabilistic retrieval framework and is an improvement over TF-IDF.

    Args:
    documents (List[Document]): List of documents to index.

    Returns:
    BM25Okapi: An index that can be used for BM25 scoring.
    """
    # Tokenize each document by splitting on whitespace
    # This is a simple approach and could be improved with more sophisticated tokenization
    tokenized_docs = [doc.page_content.split() for doc in documents]
    return BM25Okapi(tokenized_docs)

In [6]:
bm25 = create_bm25_index(cleaned_texts) # Create BM25 index from the cleaned texts (chunks)

### Define uma função que recupera semanticamente e por palavra- chave, normaliza as pontuações e obtém os documentos do topo k

In [7]:
def fusion_retrieval(vectorstore, bm25, query: str, k: int = 5, alpha: float = 0.5) -> List[Document]:
    """
    Perform fusion retrieval combining keyword-based (BM25) and vector-based search.

    Args:
    vectorstore (VectorStore): The vectorstore containing the documents.
    bm25 (BM25Okapi): Pre-computed BM25 index.
    query (str): The query string.
    k (int): The number of documents to retrieve.
    alpha (float): The weight for vector search scores (1-alpha will be the weight for BM25 scores).

    Returns:
    List[Document]: The top k documents based on the combined scores.
    """
    
    epsilon = 1e-8

    # Step 1: Get all documents from the vectorstore
    all_docs = vectorstore.similarity_search("", k=vectorstore.index.ntotal)

    # Step 2: Perform BM25 search
    bm25_scores = bm25.get_scores(query.split())

    # Step 3: Perform vector search
    vector_results = vectorstore.similarity_search_with_score(query, k=len(all_docs))
    
    # Step 4: Normalize scores
    vector_scores = np.array([score for _, score in vector_results])
    vector_scores = 1 - (vector_scores - np.min(vector_scores)) / (np.max(vector_scores) - np.min(vector_scores) + epsilon)

    bm25_scores = (bm25_scores - np.min(bm25_scores)) / (np.max(bm25_scores) -  np.min(bm25_scores) + epsilon)

    # Step 5: Combine scores
    combined_scores = alpha * vector_scores + (1 - alpha) * bm25_scores  

    # Step 6: Rank documents
    sorted_indices = np.argsort(combined_scores)[::-1]
    
    # Step 7: Return top k documents
    return [all_docs[i] for i in sorted_indices[:k]]

### Usar exemplo de caso

In [8]:
# Query
query = "What are the impacts of climate change on the environment?"

# Perform fusion retrieval
top_docs = fusion_retrieval(vectorstore, bm25, query, k=5, alpha=0.5)
docs_content = [doc.page_content for doc in top_docs]
show_context(docs_content)

![](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=all-rag-techniques--fusion-retrieval)